In [49]:
# Import necessary libraries
import torch
import gpytorch
from matplotlib import pyplot as plt
import numpy as np
import matplotlib as mpl  # Import matplotlib

# Set random seed for reproducibility
torch.manual_seed(0)
np.random.seed(0)

# Configure Matplotlib with your specified settings
import matplotlib as mpl

# Use LaTeX for rendering text in plots
mpl.rcParams['text.usetex'] = True
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.size'] = 12  # Adjust as needed
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12

# Define the original and new underlying functions
def original_function(x):
    return 2 * x

def new_function(x):
    return 4 * x

# Generate training data from the original underlying model y = 2x + noise
train_x = torch.linspace(0, 10, 50)
train_y = original_function(train_x) + torch.randn(train_x.size()) * 2  # Adding noise

# Define GP model with a linear kernel
class LinearGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(LinearGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.LinearMean(input_size=1)
        self.covar_module = gpytorch.kernels.LinearKernel()
    
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# Initialize likelihood and model
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = LinearGPModel(train_x.unsqueeze(-1), train_y, likelihood)

# Training the model
model.train()
likelihood.train()

# Use the Adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# Loss function (marginal log likelihood)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

training_iterations = 50
for i in range(training_iterations):
    optimizer.zero_grad()
    output = model(train_x.unsqueeze(-1))
    loss = -mll(output, train_y)
    loss.backward()
    optimizer.step()

# Set into eval mode
model.eval()
likelihood.eval()

# Define test points for plotting
test_x = torch.linspace(0, 12, 200)

# First new observation where the underlying function has changed
test_point1_x = torch.tensor([8.0])
test_point1_y = new_function(test_point1_x) + torch.randn(1) * 2  # y = 5x + noise

# Make predictions with the original model
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    observed_pred1 = likelihood(model(test_x.unsqueeze(-1)))
    test_point1_pred = likelihood(model(test_point1_x.unsqueeze(-1)))

# Create the first plot
fig, axs = plt.subplots(1, 2, figsize=(8.5, 4))  # Adjusted figsize for better fit

# First plot: Before adding the new point
axs[0].plot(train_x.numpy(), train_y.numpy(), 'k*', markersize=5, label=r'Training Data')
axs[0].plot(test_x.numpy(), original_function(test_x.numpy()), 'k--', label=r'Original Function $y=2x$')
axs[0].plot(test_x.numpy(), new_function(test_x.numpy()), 'g--', label=r'New Function $y=4x$')
axs[0].plot(test_x.numpy(), observed_pred1.mean.numpy(), 'b', linewidth=1.5, label='GP Mean Prediction')
axs[0].fill_between(
    test_x.numpy(),
    observed_pred1.mean.numpy() - 2 * observed_pred1.stddev.numpy(),
    observed_pred1.mean.numpy() + 2 * observed_pred1.stddev.numpy(),
    alpha=0.3,
    color='blue',
    label='Confidence Interval'
)
axs[0].plot(test_point1_x.numpy(), test_point1_y.numpy(), 'r*', markersize=8, label='New Observation')
axs[0].legend(loc='upper left')
axs[0].set_xlabel(r'x')
axs[0].set_ylabel(r'y')
axs[0].set_xlim(0, 10)
axs[0].set_ylim(-5, 70)
axs[0].set_title('Prediction Before Updating GP Model')
axs[0].grid(True)

# Update the training data with the new observation
train_x_updated = torch.cat([train_x, test_point1_x])
train_y_updated = torch.cat([train_y, test_point1_y])

# Reinitialize the model with the updated training data
model_updated = LinearGPModel(train_x_updated.unsqueeze(-1), train_y_updated, likelihood)

# Training the updated model
model_updated.train()
likelihood.train()

optimizer = torch.optim.Adam(model_updated.parameters(), lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model_updated)

for i in range(training_iterations):
    optimizer.zero_grad()
    output = model_updated(train_x_updated.unsqueeze(-1))
    loss = -mll(output, train_y_updated)
    loss.backward()
    optimizer.step()

# Set into eval mode
model_updated.eval()
likelihood.eval()

# Second new observation at a different location
test_point2_x = torch.tensor([7.0])
test_point2_y = new_function(test_point2_x) + torch.randn(1) * 2  # y = 5x + noise

# Make predictions with the updated model
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    observed_pred2 = likelihood(model_updated(test_x.unsqueeze(-1)))
    test_point2_pred = likelihood(model_updated(test_point2_x.unsqueeze(-1)))

# Second plot: After adding the new point
axs[1].plot(train_x_updated.numpy(), train_y_updated.numpy(), 'k*', markersize=5, label='Updated Training Data')
axs[1].plot(test_x.numpy(), original_function(test_x.numpy()), 'k--', label=r'Original Function $y=2x$')
axs[1].plot(test_x.numpy(), new_function(test_x.numpy()), 'g--', label=r'New Function $y=4x$')
axs[1].plot(test_x.numpy(), observed_pred2.mean.numpy(), 'b', linewidth=1.5, label='GP Mean Prediction')
axs[1].fill_between(
    test_x.numpy(),
    observed_pred2.mean.numpy() - 2 * observed_pred2.stddev.numpy(),
    observed_pred2.mean.numpy() + 2 * observed_pred2.stddev.numpy(),
    alpha=0.3,
    color='blue',
    label='Confidence Interval'
)
axs[1].plot(test_point2_x.numpy(), test_point2_y.numpy(), 'r*', markersize=8, label='New Observation')
axs[1].legend(loc='upper left')
axs[1].set_xlabel(r'x')
axs[1].set_ylabel(r'y')
axs[1].set_xlim(0, 10)
axs[1].set_ylim(-5, 70)
axs[1].set_title('Prediction After Updating GP Model')
axs[1].grid(True)

plt.tight_layout()

# Save the figure as a high-resolution PDF
plt.savefig('MotivationFailureDetection.pdf', format='pdf', bbox_inches='tight', dpi=300)

plt.show()


RuntimeError: latex was not able to process the following string:
b'1^\\\\text{st} Observation'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmpam6ajtso d8f24a2680165fb3b89c834882e9bcf2.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./d8f24a2680165fb3b89c834882e9bcf2.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/type1cm/type1cm.sty)
(/usr/share/texmf/tex/latex/cm-super/type1ec.sty
(/usr/share/texlive/texmf-dist/tex/latex/base/t1cmr.fd))
(/usr/share/texlive/texmf-dist/tex/latex/base/inputenc.sty)
(/usr/share/texlive/texmf-dist/tex/latex/geometry/geomet

Error in callback <function _draw_all_if_interactive at 0x7f6e167b8160> (for post_execute), with arguments args (),kwargs {}:


RuntimeError: latex was not able to process the following string:
b'1^\\\\text{st} Observation'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmpy61th4fm d8f24a2680165fb3b89c834882e9bcf2.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./d8f24a2680165fb3b89c834882e9bcf2.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/type1cm/type1cm.sty)
(/usr/share/texmf/tex/latex/cm-super/type1ec.sty
(/usr/share/texlive/texmf-dist/tex/latex/base/t1cmr.fd))
(/usr/share/texlive/texmf-dist/tex/latex/base/inputenc.sty)
(/usr/share/texlive/texmf-dist/tex/latex/geometry/geomet

RuntimeError: latex was not able to process the following string:
b'1^\\\\text{st} Observation'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmplwfbhd4q d8f24a2680165fb3b89c834882e9bcf2.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./d8f24a2680165fb3b89c834882e9bcf2.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/type1cm/type1cm.sty)
(/usr/share/texmf/tex/latex/cm-super/type1ec.sty
(/usr/share/texlive/texmf-dist/tex/latex/base/t1cmr.fd))
(/usr/share/texlive/texmf-dist/tex/latex/base/inputenc.sty)
(/usr/share/texlive/texmf-dist/tex/latex/geometry/geomet

<Figure size 850x400 with 2 Axes>